# 面试问题：长程 Agent 怎样设计可复现评测？`pass@k` 与 `pass^k` 有什么区别？

**回答主线。** Agent eval 要把任务、初始环境、工具、策略、预算、终态 oracle 和禁止副作用一起版本化，并运行多次 trial。`pass@k` 衡量 k 次里至少一次成功，适合“多试一次即可”；`pass^k` 衡量 k 次全部成功，适合用户期望每次可靠。只报单次平均分会隐藏非确定性与长尾失败。

下面实现状态式 sandbox、组合 grader、有限样本估计、macro/micro、paired bootstrap、trace replay、污染审计和发布门禁。受控环境用于验证评测方法，不代表真实 Agent 基准难度。


In [ ]:
import hashlib, json, math  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
from collections import Counter  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

# 固定 RNG 使 bootstrap 和模拟 trial 可复现。
rng162 = np.random.default_rng(162)  # 计算并保存当前步骤的中间状态。

def digest162(value):  # 定义本节可复用的核心函数。
    payload = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(payload.encode()).hexdigest()  # 返回当前分支计算出的结果。

assert len(digest162({"x": 1})) == 64  # 用受控断言验证关键不变量。
assert digest162({"x": 1, "y": 2}) == digest162({"y": 2, "x": 1})  # 用受控断言验证关键不变量。
assert digest162({"x": 1}) != digest162({"x": 2})  # 用受控断言验证关键不变量。


## 1. Task contract 固定初态、目标、预算和禁止副作用

模糊自然语言任务无法稳定评分。Task 记录环境 revision、初始 state、目标 predicate、允许工具、最大步数和 forbidden effects；测试私有字段不应暴露给 Agent。


In [ ]:
# Task 把自然语言目标落成可复现环境、预算和安全边界。
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Task162:  # 定义承载本节状态与行为的数据结构。
    task_id: str  # 执行当前语句以推进本节示例。
    environment_revision: str  # 执行当前语句以推进本节示例。
    initial_state: dict  # 执行当前语句以推进本节示例。
    goal: dict  # 执行当前语句以推进本节示例。
    allowed_tools: tuple  # 执行当前语句以推进本节示例。
    max_steps: int  # 执行当前语句以推进本节示例。
    forbidden_recipients: tuple = ()  # 计算并保存当前步骤的中间状态。

task162 = Task162("mail-7", "mail-sim-v3", {"drafts": [], "sent": []}, {"sent_to": "alice@example.com", "subject": "report"}, ("draft.create", "mail.send"), 5, ("mallory@example.com",))  # 计算并保存当前步骤的中间状态。
assert task162.max_steps == 5  # 用受控断言验证关键不变量。
assert "mail.send" in task162.allowed_tools  # 用受控断言验证关键不变量。
assert task162.initial_state["sent"] == []  # 用受控断言验证关键不变量。


## 2. Sandbox 执行动作并记录状态变化，不让 grader 猜文本

最终回答“已发送”不等于环境真的发送。执行器校验工具白名单和参数，返回新状态与 append-only trace；失败动作同样记录，便于区分模型错误与环境错误。


In [ ]:
def run_actions162(task, actions):  # 定义本节可复用的核心函数。
    # 深拷贝教学状态，保证每个 trial 从完全相同初态开始。
    state = json.loads(json.dumps(task.initial_state))  # 计算并保存当前步骤的中间状态。
    trace = []  # 计算并保存当前步骤的中间状态。
    for step, action in enumerate(actions):  # 遍历输入元素以累积或检查结果。
        if step >= task.max_steps:  # 按当前条件选择后续控制路径。
            trace.append({"step": step, "status": "budget_exceeded"}); break  # 执行当前语句以推进本节示例。
        tool, args = action["tool"], action["args"]  # 计算并保存当前步骤的中间状态。
        if tool not in task.allowed_tools:  # 按当前条件选择后续控制路径。
            trace.append({"step": step, "tool": tool, "status": "forbidden_tool"}); continue  # 执行当前语句以推进本节示例。
        if tool == "draft.create":  # 按当前条件选择后续控制路径。
            state["drafts"].append(dict(args)); status = "ok"  # 计算并保存当前步骤的中间状态。
        elif tool == "mail.send":  # 按当前条件选择后续控制路径。
            state["sent"].append(dict(args)); status = "ok"  # 计算并保存当前步骤的中间状态。
        trace.append({"step": step, "tool": tool, "args": dict(args), "status": status})  # 执行当前语句以推进本节示例。
    return state, trace  # 返回当前分支计算出的结果。

good_actions162 = [{"tool": "draft.create", "args": {"to": "alice@example.com", "subject": "report"}}, {"tool": "mail.send", "args": {"to": "alice@example.com", "subject": "report"}}]  # 计算并保存当前步骤的中间状态。
state162, trace162 = run_actions162(task162, good_actions162)  # 计算并保存当前步骤的中间状态。
assert len(state162["sent"]) == 1  # 用受控断言验证关键不变量。
assert all(event["status"] == "ok" for event in trace162)  # 用受控断言验证关键不变量。
assert task162.initial_state["sent"] == []  # 用受控断言验证关键不变量。


## 3. Grader 组合终态、策略与轨迹效率

Outcome grader 检查任务是否完成；policy grader 检查禁止收件人和工具；trace grader 检查预算/冗余。硬安全失败应覆盖总体成功，不能被流畅答案或部分得分抵消。


In [ ]:
def grade_trial162(task, state, trace):  # 定义本节可复用的核心函数。
    # 终态允许多种合法轨迹，但任何 forbidden side effect 都直接失败。
    sent = state.get("sent", [])  # 计算并保存当前步骤的中间状态。
    goal_ok = any(x.get("to") == task.goal["sent_to"] and x.get("subject") == task.goal["subject"] for x in sent)  # 计算并保存当前步骤的中间状态。
    forbidden = any(x.get("to") in task.forbidden_recipients for x in sent)  # 计算并保存当前步骤的中间状态。
    policy_ok = not forbidden and all(e.get("tool") in task.allowed_tools for e in trace if "tool" in e)  # 计算并保存当前步骤的中间状态。
    budget_ok = len(trace) <= task.max_steps and not any(e["status"] == "budget_exceeded" for e in trace)  # 计算并保存当前步骤的中间状态。
    return {"goal": goal_ok, "policy": policy_ok, "budget": budget_ok, "passed": goal_ok and policy_ok and budget_ok}  # 返回当前分支计算出的结果。

grade162 = grade_trial162(task162, state162, trace162)  # 计算并保存当前步骤的中间状态。
bad_state162, bad_trace162 = run_actions162(task162, [{"tool": "mail.send", "args": {"to": "mallory@example.com", "subject": "report"}}])  # 计算并保存当前步骤的中间状态。
bad_grade162 = grade_trial162(task162, bad_state162, bad_trace162)  # 计算并保存当前步骤的中间状态。
assert grade162["passed"]  # 用受控断言验证关键不变量。
assert not bad_grade162["policy"] and not bad_grade162["passed"]  # 用受控断言验证关键不变量。
assert grade162["budget"]  # 用受控断言验证关键不变量。


## 4. `pass@k` 与 `pass^k` 描述相反的产品需求

若单次成功率为 p，独立假设下 `pass@k=1-(1-p)^k` 随 k 上升，`pass^k=p^k` 随 k 下降。有限 n 次 trial 可用组合数估计抽 k 次至少一次/全部成功，避免直接代入 noisy p。


In [ ]:
def pass_metrics162(successes, k):  # 定义本节可复用的核心函数。
    # 从 n 个已观测 trial 无放回抽 k 个，计算至少一个成功与全部成功概率。
    values = np.asarray(successes, dtype=bool)  # 计算并保存当前步骤的中间状态。
    n, c = len(values), int(values.sum())  # 计算并保存当前步骤的中间状态。
    if not 1 <= k <= n: raise ValueError("k out of range")  # 按当前条件选择后续控制路径。
    all_fail = math.comb(n - c, k) / math.comb(n, k) if n - c >= k else 0.0  # 计算并保存当前步骤的中间状态。
    all_success = math.comb(c, k) / math.comb(n, k) if c >= k else 0.0  # 计算并保存当前步骤的中间状态。
    return {"pass@k": 1.0 - all_fail, "pass^k": all_success, "pass@1": c / n}  # 返回当前分支计算出的结果。

trials162 = [1, 1, 1, 0]  # 计算并保存当前步骤的中间状态。
metrics1_162 = pass_metrics162(trials162, 1)  # 计算并保存当前步骤的中间状态。
metrics2_162 = pass_metrics162(trials162, 2)  # 计算并保存当前步骤的中间状态。
assert metrics1_162["pass@1"] == 0.75  # 用受控断言验证关键不变量。
assert metrics2_162["pass@k"] > metrics1_162["pass@k"]  # 用受控断言验证关键不变量。
assert metrics2_162["pass^k"] < metrics1_162["pass^k"]  # 用受控断言验证关键不变量。


## 5. Macro task success 与 micro trial success 回答不同问题

Micro 会让 trial 更多的任务权重更大；Macro 先算每题成功率再等权平均，更接近 benchmark 任务分布。两者都应按领域、长度、工具和风险 slice 报告。


In [ ]:
def aggregate162(results_by_task):  # 定义本节可复用的核心函数。
    # macro 对任务等权，micro 对所有 trial 等权，并返回最差任务定位长尾。
    rates = {task: float(np.mean(values)) for task, values in results_by_task.items()}  # 计算并保存当前步骤的中间状态。
    macro = float(np.mean(list(rates.values())))  # 计算并保存当前步骤的中间状态。
    micro = float(np.mean([value for values in results_by_task.values() for value in values]))  # 计算并保存当前步骤的中间状态。
    return {"macro": macro, "micro": micro, "rates": rates, "worst": min(rates, key=rates.get)}  # 返回当前分支计算出的结果。

aggregate_report162 = aggregate162({"easy": [1] * 20, "hard": [0, 1]})  # 计算并保存当前步骤的中间状态。
assert aggregate_report162["macro"] == 0.75  # 用受控断言验证关键不变量。
assert aggregate_report162["micro"] > aggregate_report162["macro"]  # 用受控断言验证关键不变量。
assert aggregate_report162["worst"] == "hard"  # 用受控断言验证关键不变量。


## 6. 模型比较使用同 task/seed 的 paired bootstrap

Agent 非确定性很大；A/B 应共享 task 和环境 seed，再按 task 重采样差值。跨任务独立 bootstrap 会把任务难度差误认为模型差。


In [ ]:
def paired_bootstrap162(baseline, candidate, repeats=3000):  # 定义本节可复用的核心函数。
    # 数组每行对应同一 task 的聚合成功率，重采样行而不是单步事件。
    baseline, candidate = np.asarray(baseline, float), np.asarray(candidate, float)  # 计算并保存当前步骤的中间状态。
    if baseline.shape != candidate.shape: raise ValueError("paired shape required")  # 按当前条件选择后续控制路径。
    indices = rng162.integers(0, len(baseline), size=(repeats, len(baseline)))  # 计算并保存当前步骤的中间状态。
    sampled = (candidate - baseline)[indices].mean(axis=1)  # 计算并保存当前步骤的中间状态。
    return float((candidate - baseline).mean()), tuple(np.quantile(sampled, [0.025, 0.975]))  # 返回当前分支计算出的结果。

baseline162 = np.array([0.2, 0.8, 0.4, 0.6, 0.3, 0.5])  # 计算并保存当前步骤的中间状态。
candidate162 = np.array([0.5, 0.9, 0.6, 0.7, 0.5, 0.7])  # 计算并保存当前步骤的中间状态。
delta162, ci162 = paired_bootstrap162(baseline162, candidate162)  # 计算并保存当前步骤的中间状态。
assert delta162 > 0  # 用受控断言验证关键不变量。
assert ci162[0] <= delta162 <= ci162[1]  # 用受控断言验证关键不变量。
assert len(ci162) == 2  # 用受控断言验证关键不变量。


## 7. Trace replay 绑定环境版本、模型配置与随机种子

保存最终分数无法定位回归。Replay manifest 记录 task/env、model、prompt、tool schema、seed 和 action trace 摘要；环境发生外部变化时应使用快照或模拟器，而不是声称精确重放。


In [ ]:
def replay_manifest162(task, model_revision, prompt_revision, tool_revision, seed, trace):  # 定义本节可复用的核心函数。
    # trace digest 保护事件顺序与参数，manifest 本身再计算总摘要。
    record = {"task": task.task_id, "env": task.environment_revision, "model": model_revision, "prompt": prompt_revision, "tools": tool_revision, "seed": seed, "trace_digest": digest162(trace)}  # 计算并保存当前步骤的中间状态。
    record["manifest_digest"] = digest162(record)  # 计算并保存当前步骤的中间状态。
    return record  # 返回当前分支计算出的结果。

manifest162 = replay_manifest162(task162, "agent-v7", "prompt-v4", "tools-v2", 42, trace162)  # 计算并保存当前步骤的中间状态。
manifest_same162 = replay_manifest162(task162, "agent-v7", "prompt-v4", "tools-v2", 42, trace162)  # 计算并保存当前步骤的中间状态。
manifest_changed162 = replay_manifest162(task162, "agent-v7", "prompt-v4", "tools-v2", 42, trace162 + [{"status": "extra"}])  # 计算并保存当前步骤的中间状态。
assert manifest162 == manifest_same162  # 用受控断言验证关键不变量。
assert manifest162["trace_digest"] != manifest_changed162["trace_digest"]  # 用受控断言验证关键不变量。
assert len(manifest162["manifest_digest"]) == 64  # 用受控断言验证关键不变量。


## 8. 发布门禁加入污染、grader 与失败类型审计

Agent benchmark 容易泄漏 gold patch、隐藏测试路径或环境凭据。任务 digest 与训练索引做去重，canary task 检查 harness；失败按 planning/tool/permission/environment/policy 分类。总体提升不能覆盖安全 slice 回退。


In [ ]:
def release_gate162(candidate_macro, baseline_macro, safety_pass, contamination_hits, grader_agreement, failures):  # 定义本节可复用的核心函数。
    # 硬门禁优先于平均质量，失败直方图用于下一轮修复而非改变分数。
    histogram = Counter(failures)  # 计算并保存当前步骤的中间状态。
    passed = candidate_macro >= baseline_macro + 0.03 and safety_pass >= 0.98 and contamination_hits == 0 and grader_agreement >= 0.95  # 调整当前循环或占位控制流。
    return passed, {"delta": candidate_macro - baseline_macro, "safety": safety_pass, "contamination": contamination_hits, "grader_agreement": grader_agreement, "failures": dict(histogram)}  # 返回当前分支计算出的结果。

gate162, report162 = release_gate162(0.68, 0.62, 0.99, 0, 0.97, ["planning", "tool", "planning"])  # 计算并保存当前步骤的中间状态。
assert gate162 and report162["failures"]["planning"] == 2  # 用受控断言验证关键不变量。
assert report162["delta"] > 0.03  # 用受控断言验证关键不变量。
assert not release_gate162(0.8, 0.62, 0.9, 0, 0.97, [])[0]  # 用受控断言验证关键不变量。


## 面试总结

- Agent eval 的样本是 task + environment + tools + policy + budget + oracle，不是孤立问答。
- Outcome、policy、trace grader 分层定位；硬安全失败不能被平均分抵消。
- `pass@k` 看多次至少一次成功，`pass^k` 看连续 k 次都成功；交互产品尤其需要后者描述可靠性。
- 多 seed、paired bootstrap、trace replay、污染审计和 grader agreement 共同构成发布证据。

延伸阅读：[Demystifying Evals for AI Agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents)、[τ-bench](https://arxiv.org/abs/2406.12045)、[SWE-bench](https://arxiv.org/abs/2310.06770)、[AgentBench](https://arxiv.org/abs/2308.03688)。
